In [15]:
import os
import time
from test_gpt import OpenAI
from dotenv import load_dotenv
from pydantic import BaseModel
from anthropic import Anthropic 
import google.generativeai as gai

load_dotenv()

gai.configure(api_key=os.getenv('GEMINI_API'))

# Gemini API

In [9]:
SYSTEM_INSTRUCTION  = '''
You are a robot named BEMO. Your job is to help the user anything they ask for. You will recieve text from the user and you should reply in a JSON format. Here is an example of a request and its reply: 

request: Hey BEMO, can you add toothpaste to my shopping cart?

reply:
{
    "user's_emotion" : "neutral",
    "method" : "shop",
    "request" : "Add toothpaste to shopping cart",
}

You shold state the user's emotion based on their words, the method the user is requesting, and the the request they have asked for. The methods that you can choose from are the following:  printing (3D printing request), general question (the user is asking a general question), shop (The user is requesting to add items to their shopping cart)'''

gemini = gai.GenerativeModel('gemini-1.5-flash-8b', system_instruction=SYSTEM_INSTRUCTION)

In [4]:
request = "Bemo, whats the weather like today?"

In [23]:
start_time = time.time()
response = gemini.generate_content(request)
end_time = time.time()

print(response.text)
print(f"Time taken(Gemini): {end_time - start_time}")

{
    "user's_emotion" : "neutral",
    "method" : "general question",
    "request" : "What is the weather like today?"
} 

Time taken(Gemini): 0.6203880310058594


## Gemini 1.5-Flash-8b
### Free
#### Quotas:
- 15 RPM (Request per minute)
- 1 million TPM (Token per minute)
- 1500 RPD (Request per day)


### Pay-as-you-go
#### Quotas:
- 4000 RPM (Request per minute)
- 4 million TPM (Token per minute)
#### Price:
- Input: $0.0375 per req
- Output: $0.15 per req
- Total per request: $0.1875 per req (9.14 EGP)

## Gemini 1.5-Flash
### Free
#### Quotas:
- 15 RPM (Request per minute)
- 1 million TPM (Token per minute)
- 1500 RPD (Request per day)


### Pay-as-you-go
#### Quotas:
- 2000 RPM (Request per minute)
- 4 million TPM (Token per minute)
#### Price:
- Input: $0.075 per req
- Output: $0.3 per req
- Total per request: $0.375 per req (18.27 EGP)

# OpenAI API

In [19]:
class ResponseFormatJSONSchema(BaseModel):
    user_emotion: str | None
    method: str
    request: str
    items: list[str] | None

In [38]:
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

In [49]:
completion = client.beta.chat.completions.parse(
    model="gpt-4o-2024-08-06",
    max_tokens=50,
    messages=[
        {"role": "system", "content": "You are a helpful assistant for a user who needs help with their shopping. The user will ask you to add items to their cart, and you will need to understand what they are asking for."},
        {"role": "user", "content": "Can you add toothpaste to my cart?"}
    ],
    response_format=ResponseFormatJSONSchema,
)

In [50]:
completion.choices[0].message.content

'{"user_emotion":null,"method":"add_to_cart","request":"Add toothpaste to the cart","items":["toothpaste"]}'

In [45]:
gpt_respone = client.chat.completions.create(
    model="gpt-3.5-turbo",
    max_tokens=50,
    messages=[
        {"role": "system", "content": "You are a helpful assistant for a user who needs help with their shopping. The user will ask you to add items to their cart, and you will need to understand what they are asking for."},
        {"role": "user", "content": "Can you add toothpaste to my cart?"}
    ],
    response_format={
        "type":"json_schema",
        "json_schema":{
            "name": "respone_schema",
            "schema":
            {"type": "object",
            "properties": {
                "user's_emotion": {
                    "type": "string",
                    "enum": ["neutral", "happy", "sad", "angry"]
                },
                "method": {
                    "type": "string",
                    "enum": ["printing", "general question", "shop"]
                },
                "request": {
                    "type": "string",
                    "description": "The user's request"
                }
            },}
        }
    }
)

TypeError: Object of type ModelMetaclass is not JSON serializable

In [43]:
gpt_respone_content = gpt_respone.choices[0].message.content
print(gpt_respone_content)

Sure! I've added toothpaste to your cart. Is there anything else you would like to add?


# Claude API

In [34]:
claude_model = Anthropic(api_key=os.getenv('CLAUDE_API_KEY'))

In [40]:
claude_response = claude_model.messages.create(
    model="claude-3-5-sonnet-20241022",
    max_tokens=100,
    temperature=0.5,
    messages=[
        {"role": "system", "content": "You are a helpful assistant for a user who needs help with their shopping. The user will ask you to add items to their cart, and you will need to understand what they are asking for."},
        {"role": "user", "content": "Can you add toothpaste to my cart?"}
    ],
    system="""You are a robot assistant named BEMO who helps user with their requets by understanding their emotions, the method they are requesting, and the request they have asked for. Generate the response in a JSON format like the following example.
    request: Hey BEMO, can you add toothpaste to my shopping cart?

reply:
{
    "user's_emotion" : "neutral",
    "method" : "shop",
    "request" : "Add toothpaste to shopping cart",
}
"""
)

BadRequestError: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'Your credit balance is too low to access the Anthropic API. Please go to Plans & Billing to upgrade or purchase credits.'}}